In [1]:
# Model Evaluation Notebook
# Load trained model and evaluate with different n_steps values

import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

import json
import jax
import jax.numpy as jnp
import jax.random as jr
import optax

from config import Config
from data import load_shakespeare_dataset
from model import infer_forward_euler_with_force, logits_from_v, evaluate
from utils import load_params, generate_text

In [2]:
# Load config and model from the saved run
MODEL_DIR = "data/shakespeare/20260205_202744"

# Load run config
with open(f"{MODEL_DIR}/run_config.json", "r") as f:
    run_config = json.load(f)

print("Run config:")
for k, v in run_config.items():
    print(f"  {k}: {v}")

Run config:
  D: 256
  L: 64
  M: 256
  T_final: 0.1
  batch_size: 128
  beta: 0.1
  fast_weight_decay: 0.0
  force_penalty_duration: 10000
  force_penalty_scale: 0.05
  force_penalty_start: 20000
  learning_rate: 0.001
  lr_end_factor: 0.3
  lr_init_value: 0.0
  lr_peak_value: 0.01
  max_norm: 1.0
  max_steps: 30000
  n_steps: 10
  seed: 0
  slow_weight_decay: 5e-05
  step_size: 0.01
  tau_h: 0.01
  tau_v: 0.1
  train_epochs: 200
  use_multi_transform: False
  vocab_size: 65
  xi_attn_embed_raw_scale: 0.1
  xi_hopf_raw_scale: 0.06
  xi_pos_raw_scale: 0.1
  ctx_length: 64
  sample_mode: False
  temperature: 0.8
  gen_chars: 64
  config_file: configs/config_0.json


In [3]:
# Create Config object from saved config
# Filter out non-Config parameters
config_params = {k: v for k, v in run_config.items() 
                 if k not in ['ctx_length', 'sample_mode', 'temperature', 'gen_chars', 'config_file', 'n_steps']}

cfg = Config(**config_params)
print(f"Config created: D={cfg.D}, L={cfg.L}, M={cfg.M}, vocab_size={cfg.vocab_size}")
print(f"n_steps (default, T_final={cfg.T_final}/step_size={cfg.step_size}): {cfg.n_steps}")

# Load model parameters
params = load_params(f"{MODEL_DIR}/model_shakespeare.npz")
print(f"\nModel parameters loaded from {MODEL_DIR}/model_shakespeare.npz")
print(f"Parameter keys: {list(params.keys())}")

Config created: D=256, L=64, M=256, vocab_size=65
n_steps (default, T_final=0.1/step_size=0.01): 10

Model parameters loaded from data/shakespeare/20260205_202744/model_shakespeare.npz
Parameter keys: ['W_dec', 'a', 'b', 'b_dec', 'c', 'xi_attn_embed_raw', 'xi_hopf_raw', 'xi_pos_raw']


In [4]:
# Load Shakespeare dataset
ctx_length = run_config['ctx_length']  # 64
train_X, train_y, valid_X, valid_y, char_to_idx, idx_to_char = load_shakespeare_dataset(
    ctx_length=ctx_length,
    filename_prefix="shakespeare_data"
)

print(f"\nDataset loaded:")
print(f"  Train: {train_X.shape}, {train_y.shape}")
print(f"  Valid: {valid_X.shape}, {valid_y.shape}")

Loaded dataset: 1003797 training sequences, 111533 validation sequences
Context length: 64
Vocabulary size: 65 unique characters

Dataset loaded:
  Train: (1003797, 64), (1003797,)
  Valid: (111533, 64), (111533,)


In [13]:
train_X[0]

Array([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43,
       44, 53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39,
       52, 63,  1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1,
       51, 43,  1, 57, 54, 43, 39, 49,  8,  0,  0, 13, 50], dtype=int32)

In [15]:
jnp.ones(64, dtype=jnp.int32)

Array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],      dtype=int32)

In [23]:
# Generate 500 characters using the trained model
seed_context = train_X[0]  # Use first training example as seed
seed_text = "".join([idx_to_char[int(idx)] for idx in seed_context])

key = jr.PRNGKey(42)
generated_text = generate_text(
    params,
    seed_context,
    idx_to_char,
    cfg,
    num_chars=500,
    temperature=run_config.get('temperature', 0.8),
    key=key,
)

print("=" * 80)
print("TEXT GENERATION (500 characters)")
print("=" * 80)
print(f"\nSeed context ({len(seed_context)} chars):")
print(f"'{seed_text}'")
print(f"\nGenerated text ({len(generated_text)} chars):")
print(f"'{generated_text}'")

TEXT GENERATION (500 characters)

Seed context (64 chars):
'First Citizen:
Before we proceed any further, hear me speak.

Al'

Generated text (500 chars):
'l is again he their helen' with them rong behome am the head!

POMPEY:
I should here shall no love it youth,
The shere to jestoing maid work you ther's that did
And that warn be action to or cheer what well twalf and thy but make of your by it thou again, their their devennem, your him her shall not ruis highnes to such out be lay.

LUCIO:
Then, why you arning to she sing,
One treasulter, honough wick
Then, it and hand happear'd a ferefore at morew is he would no
be the remanls,
Is no with-dange'


In [31]:
# Generate 500 characters using the trained model
seed_context = valid_X[10]  # Use first training example as seed
seed_text = "".join([idx_to_char[int(idx)] for idx in seed_context])

key = jr.PRNGKey(2)
generated_text = generate_text(
    params,
    seed_context,
    idx_to_char,
    cfg,
    num_chars=500,
    temperature=run_config.get('temperature', 0.8),
    key=key,
)

print("=" * 80)
print("TEXT GENERATION (500 characters)")
print("=" * 80)
print(f"\nSeed context ({len(seed_context)} chars):")
print(f"'{seed_text}'")
print(f"\nGenerated text ({len(generated_text)} chars):")
print(f"'{generated_text}'")

TEXT GENERATION (500 characters)

Seed context (64 chars):
'gentleman thus grieved as I?
But who comes here?

GREMIO:
Good m'

Generated text (500 chars):
'e? I can heaven me you cannot moff,
Thy love polus net
this hund thy hear Chrow shall to the sound a shall be claing?

POMPEY:
Not a whild! But what the was mou honough sirstory and for we did who thee in't.

TYBALT:
I do be not lushfound, in the acquaenst of my hear that not, good seen heaven here prove bus mines, he counding how to rewell'd in thy compain crues at lady to sparlen.

PAULINA:
The peace,
In have to when in I hoperseizen:
Ay, nobless, nor so more re say's dis houring of thy herr'd'


In [28]:
# Define helper functions for calculating loss and accuracy with different n_steps

def calculate_loss_accuracy(params, X, y, cfg):
    """Calculate cross-entropy loss and accuracy for a dataset."""
    V_T, _ = infer_forward_euler_with_force(
        params, jnp.zeros((y.shape[0], cfg.D), jnp.float32), X, cfg
    )
    logits = logits_from_v(params, V_T)
    
    # Cross-entropy loss
    ce_losses = optax.softmax_cross_entropy_with_integer_labels(logits, y)
    loss = float(jnp.mean(ce_losses))
    
    # Accuracy
    preds = jnp.argmax(logits, axis=1)
    accuracy = float(jnp.mean((preds == y).astype(jnp.float32)))
    
    return loss, accuracy


def evaluate_with_n_steps(params, train_X, train_y, valid_X, valid_y, base_cfg, n_steps):
    """Evaluate model with a specific number of inference steps.
    
    Keeps T_final constant and adjusts step_size to achieve desired n_steps.
    n_steps = T_final / step_size  =>  step_size = T_final / n_steps
    """
    # Calculate new step_size to achieve desired n_steps while keeping T_final constant
    new_step_size = base_cfg.T_final / n_steps
    
    # Rebuild config with new step_size (T_final stays the same)
    cfg_params = {k: getattr(base_cfg, k) for k in dir(base_cfg) 
                  if not k.startswith("_") and not callable(getattr(base_cfg, k)) and k != 'n_steps'}
    cfg_params['step_size'] = new_step_size
    
    eval_cfg = Config(**cfg_params)
    print(f"Evaluating with n_steps={eval_cfg.n_steps} (T_final={eval_cfg.T_final}, step_size={eval_cfg.step_size})")
    
    # Calculate metrics on training set (use subset for efficiency)
    train_subset_size = min(10000, len(train_X))
    train_loss, train_acc = calculate_loss_accuracy(
        params, train_X[:train_subset_size], train_y[:train_subset_size], eval_cfg
    )
    
    # Calculate metrics on validation set
    valid_loss, valid_acc = calculate_loss_accuracy(params, valid_X, valid_y, eval_cfg)
    
    return {
        'n_steps': eval_cfg.n_steps,
        'step_size': eval_cfg.step_size,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'valid_loss': valid_loss,
        'valid_acc': valid_acc,
    }

In [29]:
# Evaluate with n_steps = 10 (original), 100, and 1000
n_steps_values = [10, 100, 1000]
results = []

print("=" * 80)
print("EVALUATION WITH DIFFERENT n_steps VALUES")
print("=" * 80)
print()

for n_steps in n_steps_values:
    result = evaluate_with_n_steps(params, train_X, train_y, valid_X, valid_y, cfg, n_steps)
    results.append(result)
    print(f"  Train Loss: {result['train_loss']:.4f}, Train Accuracy: {result['train_acc']:.4f}")
    print(f"  Valid Loss: {result['valid_loss']:.4f}, Valid Accuracy: {result['valid_acc']:.4f}")
    print()

EVALUATION WITH DIFFERENT n_steps VALUES

Evaluating with n_steps=10 (T_final=0.1, step_size=0.01)
  Train Loss: 1.6303, Train Accuracy: 0.5149
  Valid Loss: 1.9081, Valid Accuracy: 0.4846

Evaluating with n_steps=100 (T_final=0.1, step_size=0.001)
  Train Loss: 159647662080.0000, Train Accuracy: 0.0495
  Valid Loss: 561705975808.0000, Valid Accuracy: 0.0435

Evaluating with n_steps=1000 (T_final=0.1, step_size=0.0001)
  Train Loss: 2010486973080272896.0000, Train Accuracy: 0.0497
  Valid Loss: 16452438888133165056.0000, Valid Accuracy: 0.0437



In [30]:
# Summary comparison table
print("=" * 80)
print("SUMMARY COMPARISON (T_final=0.1 constant, varying step_size)")
print("=" * 80)
print()
print(f"{'n_steps':>10} | {'step_size':>12} | {'Train Loss':>12} | {'Train Acc':>10} | {'Valid Loss':>12} | {'Valid Acc':>10}")
print("-" * 85)

for r in results:
    print(f"{r['n_steps']:>10} | {r['step_size']:>12.6f} | {r['train_loss']:>12.4f} | {r['train_acc']:>10.4f} | {r['valid_loss']:>12.4f} | {r['valid_acc']:>10.4f}")

print()
print("Note: The model was trained with n_steps=10 (step_size=0.01).")
print("By reducing step_size while keeping T_final=0.1 constant, we get finer")
print("integration steps over the same total time, improving numerical accuracy.")

SUMMARY COMPARISON (T_final=0.1 constant, varying step_size)

   n_steps |    step_size |   Train Loss |  Train Acc |   Valid Loss |  Valid Acc
-------------------------------------------------------------------------------------
        10 |     0.010000 |       1.6303 |     0.5149 |       1.9081 |     0.4846
       100 |     0.001000 | 159647662080.0000 |     0.0495 | 561705975808.0000 |     0.0435
      1000 |     0.000100 | 2010486973080272896.0000 |     0.0497 | 16452438888133165056.0000 |     0.0437

Note: The model was trained with n_steps=10 (step_size=0.01).
By reducing step_size while keeping T_final=0.1 constant, we get finer
integration steps over the same total time, improving numerical accuracy.


In [32]:
# Generate text with different n_steps configurations (T_final constant, varying step_size)

def create_config_with_n_steps(base_cfg, n_steps):
    """Create a new config with modified step_size to achieve desired n_steps."""
    new_step_size = base_cfg.T_final / n_steps
    cfg_params = {k: getattr(base_cfg, k) for k in dir(base_cfg) 
                  if not k.startswith("_") and not callable(getattr(base_cfg, k)) and k != 'n_steps'}
    cfg_params['step_size'] = new_step_size
    return Config(**cfg_params)


# Use same seed context and random key for fair comparison
seed_context = train_X[0]
seed_text = "".join([idx_to_char[int(idx)] for idx in seed_context])

print("=" * 80)
print("TEXT GENERATION COMPARISON (T_final=0.1 constant, varying step_size)")
print("=" * 80)
print(f"\nSeed context ({len(seed_context)} chars):")
print(f"'{seed_text}'")
print()

n_steps_values = [10, 100, 1000]
generated_texts = {}

for n_steps in n_steps_values:
    eval_cfg = create_config_with_n_steps(cfg, n_steps)
    
    key = jr.PRNGKey(42)  # Same key for reproducibility
    generated = generate_text(
        params,
        seed_context,
        idx_to_char,
        eval_cfg,
        num_chars=200,  # Shorter for comparison
        temperature=run_config.get('temperature', 0.8),
        key=key,
    )
    generated_texts[n_steps] = generated
    
    print("-" * 80)
    print(f"n_steps={n_steps}, step_size={eval_cfg.step_size}")
    print("-" * 80)
    print(f"'{generated}'")
    print()

TEXT GENERATION COMPARISON (T_final=0.1 constant, varying step_size)

Seed context (64 chars):
'First Citizen:
Before we proceed any further, hear me speak.

Al'

--------------------------------------------------------------------------------
n_steps=10, step_size=0.01
--------------------------------------------------------------------------------
'l is again he their helen' with them rong behome am the head!

POMPEY:
I should here shall no love it youth,
The shere to jestoing maid work you ther's that did
And that warn be action to or cheer wha'

--------------------------------------------------------------------------------
n_steps=100, step_size=0.001
--------------------------------------------------------------------------------
'ssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssssss'

-------------------------------------------------

In [1]:
from utils import load_params

In [2]:
params = load_params("data/shakespeare/20260208_115825/model_shakespeare.npz")

W0000 00:00:1770592483.807762   37053 mps_client.cc:510] WARNING: JAX Apple GPU support is experimental and not all JAX functionality is correctly supported!
I0000 00:00:1770592483.823981   37053 service.cc:145] XLA service 0x93104d100 initialized for platform METAL (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770592483.824010   37053 service.cc:153]   StreamExecutor device (0): Metal, <undefined>
I0000 00:00:1770592483.826075   37053 mps_client.cc:406] Using Simple allocator.
I0000 00:00:1770592483.826095   37053 mps_client.cc:384] XLA backend will use up to 26800209920 bytes on device 0 for SimpleAllocator.


Metal device set to: Apple M1 Max

systemMemory: 32.00 GB
maxCacheSize: 12.48 GB



In [3]:
params

{'W_dec': Array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], dtype=float32),
 'a': Array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, n